# 05 · settlement_value research on replay v3

Notebook 04 is the **completed v2 reference** (experiment `full_development-7f132d3ee2`, tag `settlement-v2-reference`). It stays as it is: under v3 it can still display its saved results, but it cannot resume or start v2 stages.

This notebook continues with replay v3:

1. what changed from v2 to v3, and the two frozen candidates;
2. the two candidates rerun with **every new trading feature off**, on the same 94 windows and the same scenarios;
3. v2 and v3 side by side;
4. the new features, each in its own section, **disabled until you arm it**.

**Run All is safe.** Every cell that would launch simulations is disarmed: it shows what it would do and stops. A run starts only when you set its `RUN_…` switch to `True` and run that cell. Saved results are never overwritten: every run directory is named after a digest of everything that determines its results.

In [1]:
# Use source from this checkout, even when the interpreter is shared.
import sys
from pathlib import Path
_cwd = Path.cwd().resolve()
_repo_root = next(p for p in (_cwd, *_cwd.parents) if (p / 'src/tyrex_pm').is_dir())
sys.path.insert(0, str(_repo_root / 'src'))

%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from IPython.display import display

from tyrex_pm.research.strategy_lab import staged as st
from tyrex_pm.research.strategy_lab import v3
from tyrex_pm.research.strategy_replay.run import EXECUTION_CONTRACT_VERSION, RUNNER_VERSION

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
WORKERS = 4
print('replay runner:', RUNNER_VERSION, '·', EXECUTION_CONTRACT_VERSION)

replay runner: strategy_closed_loop_v3 · replay_execution_contract_v1


## 1. What changed from v2 to v3

**Execution (applies to every run, even with all features off).** v2 sent each entry as the strategy's intended *gross shares* at the limit price, with no pre-send check and no feedback to the strategy. v3 follows the live order contract:

* **Planner and fee reserve.** The order spends `min(intended debit, cap) / (1 + fee_reserve_rate)`, i.e. at most 5 / 1.02 ≈ **4.90 USDC** for a 5 USDC intended debit, rounded down to 0.01 USDC. It is an **amount-denominated FAK BUY**: it walks the asks up to the limit with that spend. Fills are therefore about 2 % smaller than in v2, at the same prices.
* **Pre-send gate.** The live `dispatch_check` runs at decision time + `decision_delay_ms` + `final_gate_delay_ms` (0 in the shipped scenarios) on the book the strategy sees: cutoff, book freshness, price moved, candidate age. A refused order is *not sent* (no POST).
* **Outcome feedback.** The strategy is told what happened (`NOT_SENT`, `ZERO_FILL`, `PARTIAL`, `FILLED`, `UNCERTAIN`). With retry off this changes nothing: the window's single attempt is spent, as in v2.
* **Provenance.** Results record the execution policy and a digest of the execution sources; experiments are new (their id includes the replay and execution-contract versions).

Regression anchor: v3 run with the v2 order contract reproduces v2 exactly (46/46 runs on a 23-window reference set; also pinned by `tests/integration/test_replay_v3_mechanics.py`).

**New optional features (all off here; section 4):** safe entry retry · managed exits (value exit, thesis stop, SL/TP) · re-entry after a confirmed sale · shared-account replay across consecutive windows.

**The two frozen candidates** (from `config/research/settlement_value_hold_baseline_v2.json`): one attempt per window, 5 USDC intended debit, hold to resolution.

In [2]:
candidates = v3.load_candidates()
display(pd.DataFrame([{'candidate': k, 'role': c.role, 'label': c.label, 'config_id': c.config_id}
                      for k, c in candidates.items()]).set_index('candidate'))

# Only the entry search differs between the two candidates.
_flat = {k: st.flat(st.effective_parameters(c.parameters))['value'] for k, c in candidates.items()}
_both = pd.DataFrame(_flat)
display(_both[_both['default'].astype(str) != _both['comparison'].astype(str)])

# The paper and live run configs must carry the same decisions with every feature off.
_check = v3.check_run_configs(candidates)
display(_check)
assert not (_check['kind'] == 'DECISION').any(), 'a run config disagrees with its candidate'

,role,label,config_id
candidate,,,
default,"provisional default: 2c, 30-150 s",TWAP | edge 2¢ | entry from 150s remaining,3b6dbbf4edbdd6d3ac771ecb964342286c391b6fa30bb6...
comparison,"comparison: 5c, 30-240 s",TWAP | edge 5¢ | entry from 240s remaining,23ca8d880ec1345c0aa13831d50823affe3253de8ca6b2...


,default,comparison
setting,,
entry.tau_max_s,150.0,240.0
entry.min_edge_per_share,0.02,0.05


,candidate,file,setting,candidate_value,config_value,kind
0,default,settlement_value_baseline_2c_150s_paper.yaml,fees.provenance,SCENARIO_ASSUMPTION_NOT_VERIFIED,MUST_MATCH_PROVIDER_FEE_SCHEDULE,label only
1,default,settlement_value_baseline_2c_150s_paper.yaml,settlement.functional_id,reference_at_close_vs_open,twap60_at_close_vs_open,label only
2,default,settlement_value_baseline_2c_150s_paper.yaml,settlement.reference_series,chainlink:btcusd:twap:60,polymarket_rtds:crypto_prices_chainlink:btc/usd,label only
3,default,settlement_value_baseline_2c_150s_paper.yaml,new features,off,off,ok
4,default,settlement_value_baseline_2c_150s_live.templat...,fees.provenance,SCENARIO_ASSUMPTION_NOT_VERIFIED,MUST_MATCH_PROVIDER_FEE_SCHEDULE,label only
5,default,settlement_value_baseline_2c_150s_live.templat...,settlement.functional_id,reference_at_close_vs_open,twap60_at_close_vs_open,label only
6,default,settlement_value_baseline_2c_150s_live.templat...,settlement.reference_series,chainlink:btcusd:twap:60,polymarket_rtds:crypto_prices_chainlink:btc/usd,label only
7,default,settlement_value_baseline_2c_150s_live.templat...,new features,off,off,ok
8,comparison,settlement_value_leader_5c_240s_paper.yaml,fees.provenance,SCENARIO_ASSUMPTION_NOT_VERIFIED,MUST_MATCH_PROVIDER_FEE_SCHEDULE,label only
9,comparison,settlement_value_leader_5c_240s_paper.yaml,settlement.functional_id,reference_at_close_vs_open,twap60_at_close_vs_open,label only


## 2. Verify the two candidates with every new feature off

* **Windows:** the same 94 development windows frozen by the v2 experiment (holdout untouched).
* **Scenarios:** `friction` (v2 stage A) and `favorable` (v2 stage C), unchanged.
* **Configurations:** the two candidates, exactly as frozen; retry off, hold to resolution, one trade per window.
* **Output:** a new `v3_verification-…` directory under `var/research/strategy_runs/staged/`. The v2 experiment is only read.

2 candidates × 94 windows × 2 scenarios = 376 runs. Check the preview first.

In [3]:
windows = v3.frozen_windows()
print(len(windows), 'frozen development windows from', v3.V2_EXPERIMENT)
display(windows.groupby('collection').agg(windows=('market_id', 'size'),
                                          first=('window_utc', 'min'), last=('window_utc', 'max')))

KEEP_TIMELINES = False   # True also saves per-second decision series (larger, slower)
verification = v3.Verification(candidates, windows, keep_timelines=KEEP_TIMELINES)
print('verification experiment:', verification.id,
      '(on disk)' if verification.record_file.exists() else '(not yet launched)')
display(verification.preview(WORKERS))

94 frozen development windows from full_development-7f132d3ee2


,windows,first,last
collection,,,
2dd3d0d125dc4a9fb5e50daf3f9bfd3e,7,2026-10-01 15:35:00+00:00,2026-10-01 16:05:00+00:00
380e44875c8e483aa4f76e17cab06c02,38,2026-10-06 14:45:00+00:00,2026-10-06 17:50:00+00:00
58a756de83c8403983a25c5609b995f1,49,2026-10-15 14:50:00+00:00,2026-10-15 18:50:00+00:00


verification experiment: v3_verification-c5fd6a9b6e (not yet launched)


,scenario,runs,done,remaining,estimate_min
0,friction,188,0,188,24
1,favorable,188,0,188,27


In [4]:
RUN_V3_VERIFICATION = False   # set True deliberately, then run this cell
if RUN_V3_VERIFICATION:
    verification.run(workers=WORKERS)
else:
    print('Verification not armed: nothing was run.')

Verification not armed: nothing was run.


## 3. v2 versus v3, side by side

Same candidates, same windows, same scenarios: every difference below comes from the execution update, not from the strategy parameters.

* **fills / wins / win rate:** filled windows, filled windows with positive proxy-labelled P&L, wins ÷ fills;
* **debit:** USDC actually spent on fills; **net P&L:** sum of proxy-labelled P&L.

The second table lists every window whose fill, side or P&L changed, with the most likely reason: *smaller spend* (fee reserve), *no fill under the amount order*, *refused by the pre-send gate*, or *filled only under v3*.

In [5]:
v2_rows = v3.v2_summaries(candidates)
v3_rows = verification.summaries()
if v3_rows.empty:
    print('No v3 verification results yet: arm section 2 first.')
    display(v3.totals(v2_rows, candidates).round(3))
else:
    comparison = v3.compare_v2_v3(v2_rows, v3_rows, candidates)
    display(comparison.round(3).T)
    changed = v3.changed_windows(v2_rows, v3_rows, candidates)
    print(len(changed), 'changed (candidate, scenario, window) rows')
    display(changed.groupby(['candidate', 'scenario_name', 'reason']).size().rename('windows'))
    display(changed.round(4))

No v3 verification results yet: arm section 2 first.


windows  entries  fills  wins  losses    debit  net_pnl  win_rate  avg_debit_per_fill
candidate  scenario_name                                                                                       
comparison favorable           94       41     41    31      10  203.949   97.584     0.756               4.974
           friction            94       36     34    26       8  158.954   80.086     0.765               4.675
default    favorable           94       44     44    35       9  217.857   74.724     0.795               4.951
           friction            94       45     36    29       7  166.385   62.319     0.806               4.622

## 4. New features (separate, disabled until armed)

Each section explains its parameters, gives an example, and has its own `RUN_…` switch. A run uses the **frozen development windows** and one scenario, and is compared with the same candidate's verification run (section 2). Numbers here are research hypotheses, not tuned or qualified settings; read them on development windows only.

Choose the candidate the feature runs are applied to:

In [6]:
FEATURE_CANDIDATE = 'default'    # 'default' (2c, 30-150 s) or 'comparison' (5c, 30-240 s)
FEATURE_SCENARIO = 'friction'
base = candidates[FEATURE_CANDIDATE]
print('feature runs apply to:', base.label, '·', FEATURE_SCENARIO)

feature runs apply to: TWAP | edge 2¢ | entry from 150s remaining · friction


### 4.1 Safe entry retry

An *attempt* is one authorized entry. After a **not-sent** refusal that may clear (stale book, price moved, account read) or an authoritative **zero fill**, a fresh attempt may follow; it re-runs the whole evaluation (signal, size, price, cutoff). An **uncertain** outcome (acceptance or fills unknown) blocks every further BUY until reconciled. A **partial fill** is a position: no top-up. The episode keeps its side; if that side loses eligibility, pursuit stops.

| parameter | meaning |
|---|---|
| `enabled` | switch (off in both candidates) |
| `max_attempts_per_episode` | total attempts including the first (user default 3, provisional) |
| `max_attempts_per_window` | ceiling across episodes in a window |
| `max_prepare_failures_per_episode`, `max_posts_per_episode`, `max_posts_per_window` | preparation failures and venue POSTs |
| `max_retry_duration_s` | elapsed budget from the first attempt |
| `min_retry_interval_ms`, `service_backoff_ms`, `service_backoff_max_ms` | waits between attempts |

In [7]:
RETRY = {'retry': {'enabled': True, 'max_attempts_per_episode': 3, 'max_attempts_per_window': 3,
                   'max_posts_per_episode': 3, 'max_posts_per_window': 3,
                   'max_retry_duration_s': 30, 'min_retry_interval_ms': 500}}
retry_run = v3.FeatureRun('retry', base, RETRY, scenario=FEATURE_SCENARIO, windows=windows)
display(retry_run.preview())

RUN_RETRY = False   # set True deliberately, then run this cell
if RUN_RETRY:
    retry_run.run(workers=WORKERS)
display(v3.feature_vs_baseline(retry_run.summaries(), verification.summaries(), base,
                               FEATURE_SCENARIO))

{'run': 'retry-default-friction-613a38013f',
 'runs': 94,
 'done': 0,
 'status': 'not started'}

,status
0,no results yet: arm section 2 and this section


### 4.2 Managed exits

`exit.policy: hold_to_resolution` (the candidates) never sells. `managed_exit` keeps the position exitable until `exit_cutoff_tau_s`; the remainder is then held to resolution (`residual_policy`).

| parameter | meaning |
|---|---|
| `value_exit_enabled`, `value_exit_margin_per_share`, `value_exit_basis` | sell when the fee-net bid beats the model's hold value (`upper` bound or `point`) by the margin |
| `thesis_stop_enabled`, `thesis_stop_probability` | declared risk rule: exit when the held side's probability falls below the threshold |
| `exit_cutoff_tau_s` | no new strategy exit below this time to close |
| `min_exit_price`, `max_exit_attempts_per_episode` | sale floor and exit submissions per episode |
| `PROTECTION` (execution policy) | the live SL/TP/trailing block, evaluated on the held bid; one exit owner, precedence SL > thesis > TP/value |

In [8]:
EXITS = {'exit': {'policy': 'managed_exit', 'value_exit_enabled': True,
                  'value_exit_margin_per_share': '0.03', 'value_exit_basis': 'upper',
                  'thesis_stop_enabled': True, 'thesis_stop_probability': 0.25,
                  'exit_cutoff_tau_s': 30}}
PROTECTION = {'protection': {'mark': 'bid', 'size_mode': 'full',
                             'stop_loss': {'style': 'stop_market_fak', 'pct_from_entry': '0.5'}}}
exit_run = v3.FeatureRun('managed_exit', base, EXITS, scenario=FEATURE_SCENARIO,
                         windows=windows, execution_policy=PROTECTION)
display(exit_run.preview())

RUN_MANAGED_EXITS = False   # set True deliberately, then run this cell
if RUN_MANAGED_EXITS:
    exit_run.run(workers=WORKERS)
display(v3.feature_vs_baseline(exit_run.summaries(), verification.summaries(), base,
                               FEATURE_SCENARIO))

{'run': 'managed_exit-default-friction-7c3054ad0b',
 'runs': 94,
 'done': 0,
 'status': 'not started'}

,status
0,no results yet: arm section 2 and this section


### 4.3 Re-entry after a confirmed sale

A new episode in the same window is allowed only after the previous position is **fully sold** (so it needs managed exits). One position at a time, never a top-up.

| parameter | meaning |
|---|---|
| `max_trades_per_window` | episodes with confirmed exposure per window |
| `rearm`, `cooldown_s` | `signal_reset`: the traded side must first become ineligible; the cooldown always applies |
| `lock_after_protective_stop`, `lock_after_thesis_stop` | a stop ends entries for the window |
| `same_side_allowed`, `opposite_side_allowed` | side rules (opposite side is research-only) |
| `min_extra_edge_per_share`, `last_entry_tau_s`, `max_window_debit_usdc` | stricter edge, latest start, window spending cap |

In [9]:
REENTRY = {**EXITS, 'reentry': {'max_trades_per_window': 2, 'rearm': 'signal_reset',
                                'cooldown_s': 15, 'last_entry_tau_s': 90,
                                'min_extra_edge_per_share': '0.01',
                                'max_window_debit_usdc': '10'}}
reentry_run = v3.FeatureRun('reentry', base, REENTRY, scenario=FEATURE_SCENARIO,
                            windows=windows, execution_policy=PROTECTION)
display(reentry_run.preview())

RUN_REENTRY = False   # set True deliberately, then run this cell
if RUN_REENTRY:
    reentry_run.run(workers=WORKERS)
display(v3.feature_vs_baseline(reentry_run.summaries(), verification.summaries(), base,
                               FEATURE_SCENARIO))

{'run': 'reentry-default-friction-63cb370b16',
 'runs': 94,
 'done': 0,
 'status': 'not started'}

,status
0,no results yet: arm section 2 and this section


### 4.4 Shared-account replay across consecutive windows

Independent window replays each start with full cash. A continuous runner does not: a held position locks its cost until the payout is proven **and redeemed** (operator step), and account limits pause entries. This replays the frozen windows **in time order through one account**:

| parameter | meaning |
|---|---|
| `initial_cash` | starting USDC |
| `max_debit_per_episode` | entries are skipped when cash is below this |
| `max_unresolved_count`, `max_unredeemed_count` | outstanding holdings / proven-but-unredeemed payouts before entries pause |
| `daily_loss_limit` | UTC-day realized losses + all open risk + the next entry |
| `max_consecutive_losses` | pause after this many losses in a row (0 = off) |
| `redemption_delay_s` | assumed operator redemption lag after the payout is provable |

It runs one window at a time (about one replay per window) and saves a JSON summary.

In [10]:
SHARED_LIMITS = {'max_debit_per_episode': '5', 'max_unresolved_count': 6,
                 'max_unredeemed_count': 6, 'daily_loss_limit': '40',
                 'max_consecutive_losses': 0, 'redemption_delay_s': 3600}
SHARED_INITIAL_CASH = '30'

RUN_SHARED_ACCOUNT = False   # set True deliberately, then run this cell
shared = v3.shared_account(base, windows, scenario=FEATURE_SCENARIO, limits=SHARED_LIMITS,
                           initial_cash=SHARED_INITIAL_CASH, run=RUN_SHARED_ACCOUNT)
if 'windows' in shared:
    print('final cash', shared['final_cash'], '· traded windows', shared['traded_windows'],
          '· skipped', shared['skipped_by_reason'])
    display(pd.DataFrame(shared['windows']).head(20))
else:
    print('Shared-account replay not armed:', shared)

Shared-account replay not armed: {'path': 'E:\\polymarket\\Tyrex_PM\\var\\research\\strategy_runs\\staged\\v3_shared_account\\default-friction-1c63a14c43.json', 'status': 'not run'}


## 5. From here to the continuous live run

* The live template for each candidate is `config/runs/settlement_value_baseline_2c_150s_live.template.yaml` (default) and `settlement_value_leader_5c_240s_live.template.yaml` (comparison). They refuse to validate until every `REQUIRED` value (trial limits, run limits, redemption wallet) is set; run **one** at a time (they share a ledger and state directory, and a wallet).
* Before launching: section 2 shows only the expected execution-driven changes; `tyrex-pm run --config <live yaml> --validate-config` and `--check-account` pass; `TYREX_POLYGON_RPC_URL` is set so committed holdings are tracked to resolution.
* Procedure and checks: `Docs/implementation/strategies/settlement_value_operator_runbook.md`, sections 3, 4 and 7.